### Elección de Algoritmos y Prevención de Data Leakage
Se implementa un **Pipeline** que integra el preprocesamiento y el modelo. Esto es crítico para asegurar la **reproducibilidad total** y evitar la fuga de información (Data Leakage) durante la validación cruzada, garantizando que el escalado de datos se realice solo con información del conjunto de entrenamiento.

Se elige **Random Forest** y **KNN** sobre modelos lineales simples debido a que las variables fisiológicas (como la relación entre IMC, edad y gasto calórico) presentan relaciones no lineales complejas que requieren modelos con mayor capacidad de captura de patrones.

In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')
from src.data_preprocessing import load_gym_data
from src.model_training import get_regression_pipeline, get_classification_pipeline
from src.model_evaluation import evaluate_model_cv

# Configuración visual
sns.set_theme(style="whitegrid")

# Cargar Datos
df = load_gym_data('../data/gym_members_exercise_tracking.csv')

# Preparar variables
X_reg = df.drop(columns=['Fat_Percentage'])
y_reg = df['Fat_Percentage']

X_clf = df.drop(columns=['Experience_Level'])
y_clf = df['Experience_Level']

# --- 1. EVALUACIÓN DE REGRESIÓN ---
print("=== Resultados: Predicción de Porcentaje de Grasa (Regresión) ===")
# Instanciar modelos
lin_reg = get_regression_pipeline('linear')
rf_reg = get_regression_pipeline('random_forest')
knn_reg_euc = get_regression_pipeline('knn', knn_metric='euclidean')
knn_reg_man = get_regression_pipeline('knn', knn_metric='manhattan')

# Evaluar con validación cruzada
res_lin = evaluate_model_cv(lin_reg, X_reg, y_reg, task='regression', cv=5)
res_rf_reg = evaluate_model_cv(rf_reg, X_reg, y_reg, task='regression', cv=5)
res_knn_euc = evaluate_model_cv(knn_reg_euc, X_reg, y_reg, task='regression', cv=5)
res_knn_man = evaluate_model_cv(knn_reg_man, X_reg, y_reg, task='regression', cv=5)

# Concatenar para comparar
df_reg_results = pd.concat([res_lin, res_rf_reg, res_knn_euc, res_knn_man], axis=1)
df_reg_results.columns = ['Regresión Lineal', 'Random Forest Reg.', 'KNN (Pitágoras)', 'KNN (Manhattan)']
display(df_reg_results)

# --- 2. EVALUACIÓN DE CLASIFICACIÓN ---
print("\n=== Resultados: Predicción de Nivel de Experiencia (Clasificación) ===")
# Instanciar modelos
log_reg = get_classification_pipeline('logistic')
rf_clf = get_classification_pipeline('random_forest')
knn_clf_euc = get_classification_pipeline('knn', knn_metric='euclidean')
knn_clf_man = get_classification_pipeline('knn', knn_metric='manhattan')

# Evaluar con validación cruzada
res_log = evaluate_model_cv(log_reg, X_clf, y_clf, task='classification', cv=5)
res_rf_clf = evaluate_model_cv(rf_clf, X_clf, y_clf, task='classification', cv=5)
res_knn_clf_euc = evaluate_model_cv(knn_clf_euc, X_clf, y_clf, task='classification', cv=5)
res_knn_clf_man = evaluate_model_cv(knn_clf_man, X_clf, y_clf, task='classification', cv=5)

# Concatenar para comparar
df_clf_results = pd.concat([res_log, res_rf_clf, res_knn_clf_euc, res_knn_clf_man], axis=1)
df_clf_results.columns = ['Regresión Logística', 'Random Forest Clf.', 'KNN (Pitágoras)', 'KNN (Manhattan)']
display(df_clf_results)

=== Resultados: Predicción de Porcentaje de Grasa (Regresión) ===


,Regresión Lineal,Random Forest Reg.,KNN (Pitágoras),KNN (Manhattan)
fit_time,0.006750,0.121217,0.005324,0.005119
score_time,0.003919,0.028620,0.518007,0.007414
test_neg_mean_squared_error,-16.444341,-8.085465,-11.716240,-11.177146
test_r2,0.576027,0.791284,0.697410,0.711343
test_neg_mean_absolute_error,-3.378304,-2.393476,-2.798088,-2.742236



=== Resultados: Predicción de Nivel de Experiencia (Clasificación) ===


,Regresión Logística,Random Forest Clf.,KNN (Pitágoras),KNN (Manhattan)
fit_time,0.013666,0.139456,0.005409,0.005423
score_time,0.009126,0.033934,0.020761,0.011392
test_accuracy,0.873587,0.880788,0.793444,0.789305
test_f1_macro,0.892098,0.899369,0.821827,0.817591
test_precision_macro,0.896957,0.906270,0.826357,0.818418
test_recall_macro,0.890985,0.899392,0.822478,0.820667
